In [1]:
import pandas as pd
import numpy as np

# 1. Load the dataset
df = pd.read_csv('../data/riyasewana_raw_final.csv')
print(f"Original shape before cleaning: {df.shape}")

# 2. Filter out Motorcycles & Scooters
# Cars don't have a 'Start Type' (Kick/Electric) on Riyasewana, and usually have Engine CC > 500.
# We also ensure 'Gear' is not NaN (since bikes often lack this field in car searches).
df = df[df['Gear'].notna()]
df = df[df['Start Type'].isna()]

# 3. Clean 'Price' (The Target Variable)
# Drop "Negotiable" rows, as they don't give us a target price to train on
df = df[~df['Price'].astype(str).str.contains('Negotiable', case=False, na=False)]
df.dropna(subset=['Price'], inplace=True)

# Remove 'Rs.', commas, and convert to integer
df['Price'] = df['Price'].astype(str).str.replace('Rs.', '', regex=False)
df['Price'] = df['Price'].str.replace(',', '', regex=False).str.strip()
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

# 4. Clean 'Mileage'
# Replace hyphens '-' (often used for brand new cars) with NaN temporarily
df['Mileage'] = df['Mileage'].astype(str).replace('-', np.nan)
# Remove 'km' and commas
df['Mileage'] = df['Mileage'].str.replace('km', '', regex=False)
df['Mileage'] = df['Mileage'].str.replace(',', '', regex=False).str.strip()
df['Mileage'] = pd.to_numeric(df['Mileage'], errors='coerce')

# For Brand New/Unregistered cars with missing mileage, we can impute a low value (e.g., 50 km)
df['Mileage'] = df['Mileage'].fillna(50)

# 5. Clean 'Engine (cc)'
df['Engine (cc)'] = pd.to_numeric(df['Engine (cc)'], errors='coerce')

# 6. Feature Engineering: Vehicle Age
# Since your project is taking place in 2026, we calculate age relative to the current year
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df['Vehicle_Age'] = 2026 - df['Year']

# 7. Drop unnecessary columns
# We drop 'URL', 'Ad Date', and 'Start Type' as they hold no predictive value for the model
columns_to_drop = ['URL', 'Ad Date', 'Start Type']
df.drop(columns=[col for col in columns_to_drop if col in df.columns], inplace=True)

# 8. Final Drop of any remaining missing essential values
df.dropna(subset=['Price', 'Year', 'Make', 'Model', 'Engine (cc)'], inplace=True)

# Reset the index after all the row droppings
df.reset_index(drop=True, inplace=True)

print(f"Cleaned shape ready for training: {df.shape}")
print(df.head())

# Save the pristine dataset
df.to_csv('../data/riyasewana_processed_ready.csv', index=False)

Original shape before cleaning: (440, 13)
Cleaned shape ready for training: (321, 11)
      Price   Location  Year   Mileage      Make                   Model  \
0   8675000  Kadawatha  2023      50.0    Toyota                   Passo   
1  11900000   Matugama  2016  137000.0    Toyota                   Prius   
2   6850000    Gampaha  2024    2600.0  Daihatsu          MIRA X LTD SA3   
3  32500000   Kelaniya  2019   16000.0    Jaguar  F-Pace Highest Spec25t   
4    975000      Kandy  1985     111.0    Toyota                  Corona   

        Gear Fuel Type  Engine (cc)                   Condition  Vehicle_Age  
0  Automatic    Petrol       1000.0  Unregistered (Recondition)            3  
1  Automatic    Hybrid       1800.0           Registered (Used)           10  
2  Automatic    Petrol        658.0  Unregistered (Recondition)            2  
3  Automatic    Petrol       2000.0           Registered (Used)            7  
4     Manual    Petrol       1500.0           Registered (Used